In [2]:
import os
%pwd

'c:\\Users\\ASUS\\Desktop\\Personal_Project\\Kidney Disease Classification\\KIdney-Disease-Classification-System-using-CNN\\research'

In [4]:
os.chdir("../")

In [7]:
%pwd

'c:\\Users\\ASUS\\Desktop\\Personal_Project\\Kidney Disease Classification\\KIdney-Disease-Classification-System-using-CNN'

In [6]:
os.chdir("KIdney-Disease-Classification-System-using-CNN")

In [8]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
from cnnClassifier.constants import *## this are imported values 
from cnnClassifier.utils.common import read_yaml,create_directories
class ConfigurationManager:
    def __init__(## constuctor
        self,
        config_filepath=CONFIG_FILE_PATH,## this variable will have the file paths stored 
        params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)## this variable will give access to hard coded values 
        self.params = read_yaml(params_filepath)## read the file and return in key values pairs

        create_directories([self.config.artifacts_root])## self defined function having a file path and directary created

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])## root directary is created
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config### this have multiple data paths returned

In [ ]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size
import shutil

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def flatten_folder(self):
        base_path = self.config.unzip_dir
    
        dirs = os.listdir(base_path)
    
        # If only one folder inside → likely nested
        if len(dirs) == 1:
            inner_path = os.path.join(base_path, dirs[0])
    
            if os.path.isdir(inner_path):
                inner_items = os.listdir(inner_path)
    
                for item in inner_items:
                    src = os.path.join(inner_path, item)
                    dest = os.path.join(base_path, item)
                    shutil.move(src, dest)
    
                os.rmdir(inner_path)
    
                print("✅ Folder structure fixed!")

    def download_file(self) -> str:
        '''
        Fetch data from the url and store it in the zip file then later in the pipeline got unzipped
        '''
    
        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
    
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
    
            # ✅ ADD THIS CHECK
            if os.path.exists(zip_download_dir):
                logger.info(f"File already exists at {zip_download_dir}. Skipping download.")
                return zip_download_dir
    
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")
    
            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
    
            gdown.download(prefix + file_id, zip_download_dir)
    
            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
    
            return zip_download_dir
    
        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [ ]:
try:
    logger.info("🔷 Starting Data Ingestion Pipeline")

    # Create config manager
    config = ConfigurationManager()## this will extract the file and direcatry paths

    # Get ingestion config
    data_ingestion_config = config.get_data_ingestion_config()## path and directary paths are returnred

    # Create ingestion object
    data_ingestion = DataIngestion(config=data_ingestion_config)

    # Step 1: Download data
    data_ingestion.download_file()

    # Step 2: Extract data
    data_ingestion.extract_zip_file()

    # Step 3: Fix folder structure (IMPORTANT)
    data_ingestion.flatten_folder()

    logger.info("✅ Data Ingestion Completed Successfully")

except Exception as e:
    logger.exception(e)
    raise e

[2026-03-25 00:51:31,085: INFO: 864677484: 🔷 Starting Data Ingestion Pipeline]
[2026-03-25 00:51:31,085: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-25 00:51:31,085: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-25 00:51:31,091: INFO: common: created directory at: artifacts]
[2026-03-25 00:51:31,092: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-25 00:51:31,093: INFO: 2431983755: Downloading data from https://drive.google.com/file/d/1FccmEGzsAOs7SGWi_zxL-nURysOITNnX/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1FccmEGzsAOs7SGWi_zxL-nURysOITNnX
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1FccmEGzsAOs7SGWi_zxL-nURysOITNnX&confirm=t&uuid=4d766f80-81c8-4c63-a348-0872eb645209
To: c:\Users\ASUS\Desktop\Personal_Project\Kidney Disease Classification\KIdney-Disease-Classification-System-using-CNN\artifacts\data_ingestion\data.zip
100%|██████████| 1.63G/1.63G [07:52<00:00, 3.45MB/s]

[2026-03-25 00:59:27,145: INFO: 2431983755: Downloaded data from https://drive.google.com/file/d/1FccmEGzsAOs7SGWi_zxL-nURysOITNnX/view?usp=sharing into file artifacts/data_ingestion/data.zip]


[2026-03-25 00:59:42,171: INFO: 864677484: ✅ Data Ingestion Completed Successfully]
